In [3]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time

from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.linear_model import ElasticNet

warnings.filterwarnings("ignore")

print("="*80)
print("HTC模型性能对比验证：With vs Without RFE")
print("✨ 公平epoch对比 + 完整训练历史数据（含Loss）")
print("✨ 模型B最大训练上限：1000000 epoch")
print("✨ Excel保存：全部epoch + 每50采样 + 每55采样")
print("="*80)

# ====================== 1. 读取数据 ======================
df = pd.read_excel(r"D:\文成数据库\Nb-Si数据库2.28-高温压缩.xlsx")

features_name = [
    "Nb", "Si", "Ti", "Al", "Cr", "Hf", "Zr", "Mo", "Fe", "B", "V", 
    "VEC", "x", "Δx", "ΔHmix", "ΔSmix", "ΔG", "Tm", "ΔTm", "ρ", "r", "am", "Δa", "δ", "Ω", "λ",
    "Mean_A1 atomic number", "Mean_A6 atomic weight", "Mean_E2 electronegativity (Pauling)",
    "Mean_E5 energy ionization first", "Mean_E13 Electron affinity", "Mean_Electrophilicity Index",
    "Mean_Speed of Sound", "Mean_Neutron Cross Section", "Mean_Neutron Mass Absorption",
    "Mean_Space Group Number", "Mean_Glawe Number", "Mean_C1 temperature melting",
    "Mean_C2 temperature boiling", "Mean_density", "Mean_C3 enthalpy vaporization",
    "Mean_C4 enthalpy melting", "Mean_C5 enthalpy atomization", "Mean_热导率W/(mk)",
    "Mean_电导率(MS/m)", "Mean_电阻率(mΩ)", "Mean_热膨胀(1/k)", "Mean_比热容J/g/k",
    "Mean_摩尔热容(J/mol/k)", "Mean_摩尔体积(cm3/mol)", "Mean_莫氏硬度(MPa)",
    "Mean_covalent radius Cordero (pm)", "Mean_covalent radius Pyykko(Single Bond) (pm)",
    "Mean_covalent radius Pyykko(Double Bond) (pm)", "Mean_Pyykko(Triple Bond) (pm)",
    "Mean_S10 Lattice Constants a", "Mean_S11 Lattice Constants b", "Mean_S12 Lattice Constants c",
    "Mean_S13 radii atomic (coordination number 12) (pm)", "Mean_metallic radius(pm)",
    "Mean_metallic radius 12 Neighbors(pm)",
    "Var_A1 atomic number", "Var_A6 atomic weight", "Var_E2 electronegativity (Pauling)",
    "Var_E5 energy ionization first", "Var_E13 Electron affinity", "Var_Electrophilicity Index",
    "Var_Speed of Sound", "Var_Neutron Cross Section", "Var_Neutron Mass Absorption",
    "Var_Space Group Number", "Var_Glawe Number", "Var_C1 temperature melting",
    "Var_C2 temperature boiling", "Var_density", "Var_C3 enthalpy vaporization",
    "Var_C4 enthalpy melting", "Var_C5 enthalpy atomization", "Var_热导率W/(mk)",
    "Var_电导率(MS/m)", "Var_电阻率(mΩ)", "Var_热膨胀(1/k)", "Var_比热容J/g/k",
    "Var_摩尔热容(J/mol/k)", "Var_摩尔体积(cm3/mol)", "Var_莫氏硬度(MPa)",
    "Var_covalent radius Cordero (pm)", "Var_covalent radius Pyykko(Single Bond) (pm)",
    "Var_covalent radius Pyykko(Double Bond) (pm)", "Var_Pyykko(Triple Bond) (pm)",
    "Var_S10 Lattice Constants a", "Var_S11 Lattice Constants b", "Var_S12 Lattice Constants c",
    "Var_S13 radii atomic (coordination number 12) (pm)", "Var_metallic radius(pm)",
    "Var_metallic radius 12 Neighbors(pm)",
]

target_col = 'high temperature compression'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

X_all = df[features_name].values
y_all = df[target_col].values
y_all_np = np.array(y_all, dtype=float)

print(f"\n原始数据信息:")
print(f"  样本数: {len(y_all_np)}")
print(f"  原始特征数: {len(features_name)}")
print(f"  目标变量范围: [{np.min(y_all_np):.4f}, {np.max(y_all_np):.4f}]")

# 创建输出目录
output_dir = r"D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\带不带RFE\HTC_performance_comparison-1.25.2"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"\n创建输出目录: {output_dir}")

# ====================== 2. 特征选择流程 ======================
print("\n" + "="*80)
print("步骤1: 特征选择流程（获得Without RFE特征和6特征）")
print("="*80)

# 标准化
scaler_for_selection = StandardScaler()
X_all_scaled = scaler_for_selection.fit_transform(X_all)
n_samples, n_features = X_all_scaled.shape

# PCC去冗余
print("\n[1/3] PCC去除冗余特征...")
pcc_matrix = np.zeros((n_features, n_features))
for i in range(n_features):
    for j in range(i + 1, n_features):
        pcc_val, _ = pearsonr(X_all_scaled[:, i], X_all_scaled[:, j])
        pcc_matrix[i, j] = pcc_val
        pcc_matrix[j, i] = pcc_val

pcc_threshold = 0.95
redundant_features = set()

for i in range(n_features):
    for j in range(i + 1, n_features):
        if abs(pcc_matrix[i, j]) > pcc_threshold:
            pcc_i, _ = pearsonr(X_all_scaled[:, i], y_all_np)
            pcc_j, _ = pearsonr(X_all_scaled[:, j], y_all_np)
            if abs(pcc_i) < abs(pcc_j):
                redundant_features.add(i)
            else:
                redundant_features.add(j)

non_redundant_indices = [i for i in range(n_features) if i not in redundant_features]
X_nonredundant = X_all_scaled[:, non_redundant_indices]
nr_features_name = [features_name[i] for i in non_redundant_indices]

print(f"  去冗余前: {n_features} 个特征")
print(f"  去冗余后: {len(nr_features_name)} 个特征")

# PCC + MIC 过滤
print("\n[2/3] PCC + MIC 过滤...")
corr_threshold = 0.09
pcc_with_target = []
for i in range(X_nonredundant.shape[1]):
    p_val, _ = pearsonr(X_nonredundant[:, i], y_all_np)
    pcc_with_target.append(abs(p_val))

pcc_indices = [i for i, v in enumerate(pcc_with_target) if v > corr_threshold]

mic_values = mutual_info_regression(X_nonredundant, y_all_np)
mic_threshold = 0.08
mic_indices = [i for i, v in enumerate(mic_values) if v > mic_threshold]

selected_indices_filter = set(pcc_indices).intersection(set(mic_indices))
selected_indices_filter = sorted(list(selected_indices_filter))

filtered_features_without_rfe = [nr_features_name[i] for i in selected_indices_filter]
X_filter_without_rfe = X_nonredundant[:, selected_indices_filter]

print(f"  PCC+MIC过滤后: {len(filtered_features_without_rfe)} 个特征")
print(f"  这就是 'Without RFE' 的特征集")

# RFE选择6个特征
print("\n[3/3] RFE选择6个特征...")
n_features_to_select = 6
elasticnet_estimator = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000, random_state=0)
selector = RFE(estimator=elasticnet_estimator, n_features_to_select=n_features_to_select)
selector = selector.fit(X_filter_without_rfe, y_all_np)

selected_mask_rfe = selector.support_
final_features_6 = [filtered_features_without_rfe[i] for i, sel in enumerate(selected_mask_rfe) if sel]

print(f"  RFE选择后: {len(final_features_6)} 个特征")
print(f"  这就是 'With RFE' 的6个特征")
print(f"  特征名称: {final_features_6}")

print(f"\n✅ 特征选择完成！")
print(f"   模型A (Without RFE): {len(filtered_features_without_rfe)} 个特征")
print(f"   模型B (With RFE): {len(final_features_6)} 个特征")

# ====================== 3. 准备两组数据 ======================
print("\n" + "="*80)
print("步骤2: 准备两组数据并标准化")
print("="*80)

# Without RFE数据
X_without_rfe_original = df[filtered_features_without_rfe].values
scaler_without_rfe = StandardScaler()
X_without_rfe_scaled = scaler_without_rfe.fit_transform(X_without_rfe_original)

n_features_without_rfe = X_without_rfe_scaled.shape[1]
print(f"\n模型A数据 (Without RFE):")
print(f"  特征数: {n_features_without_rfe}")
print(f"  样本数: {X_without_rfe_scaled.shape[0]}")

# With RFE数据（6特征）
X_with_rfe_original = df[final_features_6].values
scaler_with_rfe = StandardScaler()
X_with_rfe_scaled = scaler_with_rfe.fit_transform(X_with_rfe_original)

print(f"\n模型B数据 (With RFE):")
print(f"  特征数: {X_with_rfe_scaled.shape[1]}")
print(f"  样本数: {X_with_rfe_scaled.shape[0]}")

# ====================== 4. 定义模型（HTC架构：32→16→1）======================
class HTCModel(nn.Module):
    """
    HTC模型架构：输入 → 32 → 16 → 1
    使用BatchNorm和Dropout以保持稳定性
    """
    def __init__(self, input_dim):
        super(HTCModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 32)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.05)
        
        self.layer2 = nn.Linear(32, 16)
        self.bn2 = nn.BatchNorm1d(16)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.02)
        
        self.layer3 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        x = self.layer3(x)
        return x

# ====================== 5. 组合损失函数 ======================
class CombinedLoss(nn.Module):
    def __init__(self, mse_weight=0.6, huber_weight=0.4, huber_delta=1.0):
        super(CombinedLoss, self).__init__()
        self.mse_weight = mse_weight
        self.huber_weight = huber_weight
        self.mse_loss = nn.MSELoss()
        self.huber_loss = nn.HuberLoss(delta=huber_delta)
        
    def forward(self, pred, target):
        mse = self.mse_loss(pred, target)
        huber = self.huber_loss(pred, target)
        return self.mse_weight * mse + self.huber_weight * huber

# ====================== 6. Warmup Cosine Scheduler ======================
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, base_lr, min_lr=1e-8):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_step = 0
        
    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            lr = self.base_lr * (self.current_step / self.warmup_steps)
        else:
            progress = (self.current_step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            lr = self.min_lr + (self.base_lr - self.min_lr) * 0.5 * (1 + np.cos(np.pi * progress))
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
    
    def get_last_lr(self):
        return [param_group['lr'] for param_group in self.optimizer.param_groups]

# ====================== 7. 训练函数 ======================
def train_model(X_scaled, y, model_name, n_features, max_epochs=1000000, 
                target_train_r2=0.92, target_test_r2=0.92, 
                target_train_rmse=6.0, target_test_rmse=6.0,
                force_max_epochs=False):
    """
    训练HTC模型直到达到目标性能
    ✨ 保存每个epoch的完整数据（R², RMSE, MAE, Loss）
    ✨ 支持强制训练到指定epoch数（用于公平对比）
    
    参数:
        max_epochs: 最大训练轮数（默认1000000）
        force_max_epochs: 如果为True，则训练到max_epochs而不提前停止
    """
    print(f"\n" + "-"*80)
    print(f"训练 {model_name}")
    print(f"最大训练轮数: {max_epochs}")
    print("-"*80)
    
    model = HTCModel(n_features).to(device)
    criterion = CombinedLoss(mse_weight=0.6, huber_weight=0.4, huber_delta=1.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0015, weight_decay=1e-6)
    
    warmup_steps = 200
    scheduler = WarmupCosineScheduler(optimizer, warmup_steps, max_epochs, 
                                     base_lr=0.0015, min_lr=1e-8)
    
    print(f"模型结构:")
    print(f"  输入: {n_features} 特征")
    print(f"  隐藏层1: 32 神经元 (Linear + BatchNorm + ReLU + Dropout 0.05)")
    print(f"  隐藏层2: 16 神经元 (Linear + BatchNorm + ReLU + Dropout 0.02)")
    print(f"  输出: 1 神经元")
    print(f"训练目标: Train R²>{target_train_r2}, Test R²>{target_test_r2}, "
          f"Train RMSE<{target_train_rmse}, Test RMSE<{target_test_rmse}")
    
    # 创建历史记录字典（包含loss）
    history = {
        'epoch': [],
        'train_r2': [],
        'test_r2': [],
        'train_rmse': [],
        'test_rmse': [],
        'train_mae': [],
        'test_mae': [],
        'train_loss': [],
        'test_loss': []
    }
    
    best_metrics = None
    best_random_seed = None
    consecutive_good = 0
    
    start_time = time.time()
    
    # 使用固定的随机种子以保证可比性
    np.random.seed(42)
    random_seeds = np.random.randint(0, 2147483647, size=max_epochs, dtype=np.int32)
    
    for epoch in range(max_epochs):
        seed = int(random_seeds[epoch])
        
        # 划分数据
        train_indices, test_indices = train_test_split(
            np.arange(len(X_scaled)), test_size=0.2, random_state=seed
        )
        
        X_train = X_scaled[train_indices]
        y_train = y[train_indices]
        X_test = X_scaled[test_indices]
        y_test = y[test_indices]
        
        # 转换为PyTorch张量
        train_x = torch.from_numpy(X_train.astype(np.float32)).to(device)
        train_y = torch.from_numpy(y_train.astype(np.float32)).to(device).view(-1, 1)
        test_x = torch.from_numpy(X_test.astype(np.float32)).to(device)
        test_y = torch.from_numpy(y_test.astype(np.float32)).to(device).view(-1, 1)
        
        # 训练
        model.train()
        optimizer.zero_grad()
        outputs = model(train_x)
        loss = criterion(outputs, train_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        # 评估（每个epoch都评估并保存）
        if epoch % 1 == 0:
            model.eval()
            with torch.no_grad():
                # 训练集性能
                outputs_train = model(train_x)
                train_pred = outputs_train.cpu().numpy().flatten()
                train_true = y_train.flatten()
                r2_train = r2_score(train_true, train_pred)
                rmse_train = np.sqrt(mean_squared_error(train_true, train_pred))
                mae_train = mean_absolute_error(train_true, train_pred)
                train_loss = criterion(outputs_train, train_y).item()
                
                # 测试集性能
                outputs_test = model(test_x)
                test_pred = outputs_test.cpu().numpy().flatten()
                test_true = y_test.flatten()
                r2_test = r2_score(test_true, test_pred)
                rmse_test = np.sqrt(mean_squared_error(test_true, test_pred))
                mae_test = mean_absolute_error(test_true, test_pred)
                test_loss = criterion(outputs_test, test_y).item()
                
                # 保存历史数据
                history['epoch'].append(epoch)
                history['train_r2'].append(r2_train)
                history['test_r2'].append(r2_test)
                history['train_rmse'].append(rmse_train)
                history['test_rmse'].append(rmse_test)
                history['train_mae'].append(mae_train)
                history['test_mae'].append(mae_test)
                history['train_loss'].append(train_loss)
                history['test_loss'].append(test_loss)
                
                # 每1000轮打印一次
                if epoch % 1000 == 0:
                    print(f"  Epoch {epoch:5d} | Train R²: {r2_train:.4f} | "
                          f"Test R²: {r2_test:.4f} | "
                          f"Train RMSE: {rmse_train:.4f} | Test RMSE: {rmse_test:.4f}")
                
                # 检查是否达到目标（早停逻辑）
                if (r2_train > target_train_r2 and r2_test > target_test_r2 and
                    rmse_train < target_train_rmse and rmse_test < target_test_rmse):
                    consecutive_good += 1
                    
                    if not force_max_epochs and consecutive_good >= 5:
                        elapsed_time = time.time() - start_time
                        print(f"\n  ✅ 达到目标! Epoch {epoch} (连续5次达标)")
                        print(f"     训练时间: {elapsed_time:.2f} 秒")
                        print(f"     Train R²: {r2_train:.6f} | Test R²: {r2_test:.6f}")
                        print(f"     Train RMSE: {rmse_train:.6f} | Test RMSE: {rmse_test:.6f}")
                        print(f"     Train MAE: {mae_train:.6f} | Test MAE: {mae_test:.6f}")
                        
                        best_metrics = {
                            'train_r2': r2_train,
                            'test_r2': r2_test,
                            'train_rmse': rmse_train,
                            'test_rmse': rmse_test,
                            'train_mae': mae_train,
                            'test_mae': mae_test,
                            'train_pred': train_pred,
                            'train_true': train_true,
                            'test_pred': test_pred,
                            'test_true': test_true,
                            'final_epoch': epoch,
                            'training_time': elapsed_time
                        }
                        best_random_seed = seed
                        break
                else:
                    consecutive_good = 0
    
    # 如果是强制模式或未达到目标，使用最后一个epoch的结果
    if best_metrics is None or force_max_epochs:
        elapsed_time = time.time() - start_time
        if force_max_epochs:
            print(f"\n  ✅ 强制训练完成! Epoch {epoch}")
        else:
            print(f"\n  ⚠️ 未在{max_epochs}轮内达到目标")
        print(f"     训练时间: {elapsed_time:.2f} 秒")
        print(f"     最终 Train R²: {r2_train:.4f} | Test R²: {r2_test:.4f}")
        best_metrics = {
            'train_r2': r2_train,
            'test_r2': r2_test,
            'train_rmse': rmse_train,
            'test_rmse': rmse_test,
            'train_mae': mae_train,
            'test_mae': mae_test,
            'train_pred': train_pred,
            'train_true': train_true,
            'test_pred': test_pred,
            'test_true': test_true,
            'final_epoch': epoch,
            'training_time': elapsed_time
        }
        best_random_seed = seed
    
    return model, best_metrics, train_indices, test_indices, best_random_seed, history

# ====================== 8. 训练两个模型 ======================
print("\n" + "="*80)
print("步骤3: 训练两个模型（✨ 先训练With RFE，然后用相同epoch训练Without RFE）")
print("="*80)

# 先训练模型B (With RFE - 6特征)
print("\n" + "="*60)
print("第一步：训练 Model B (With RFE - 6 features)")
print("="*60)
model_B, metrics_B, train_idx_B, test_idx_B, seed_B, history_B = train_model(
    X_with_rfe_scaled, y_all_np, 
    "Model B (With RFE - 6 features)", 
    n_features=6,
    max_epochs=1000000,
    target_train_r2=0.92,
    target_test_r2=0.92,
    target_train_rmse=6.0,
    target_test_rmse=6.0
)

# 获取模型B达到目标的epoch数
target_epochs = metrics_B['final_epoch']
print(f"\n✅ 模型B在 epoch {target_epochs} 达到目标")
print(f"   现在将训练模型A到相同的 epoch {target_epochs}")

# 然后训练模型A (Without RFE)，使用相同的epoch数
print("\n" + "="*60)
print(f"第二步：训练 Model A (Without RFE - {n_features_without_rfe} features)")
print(f"         训练到 epoch {target_epochs}（与模型B相同）")
print("="*60)
model_A, metrics_A, train_idx_A, test_idx_A, seed_A, history_A = train_model(
    X_without_rfe_scaled, y_all_np, 
    f"Model A (Without RFE - {n_features_without_rfe} features)", 
    n_features=n_features_without_rfe,
    max_epochs=target_epochs + 1,
    target_train_r2=0.92,
    target_test_r2=0.92,
    target_train_rmse=6.0,
    target_test_rmse=6.0,
    force_max_epochs=True
)

print(f"\n✅ 训练完成！")
print(f"   模型B (With RFE): {len(history_B['epoch'])} 个数据点")
print(f"   模型A (Without RFE): {len(history_A['epoch'])} 个数据点")
print(f"   两个模型训练epoch数相同: {target_epochs}")

# ====================== 9. 性能对比分析 ======================
print("\n" + "="*80)
print("步骤4: 性能对比分析")
print("="*80)

# 计算性能差异
r2_diff = metrics_B['test_r2'] - metrics_A['test_r2']
rmse_diff = metrics_B['test_rmse'] - metrics_A['test_rmse']
mae_diff = metrics_B['test_mae'] - metrics_A['test_mae']

feature_reduction = (n_features_without_rfe - 6) / n_features_without_rfe * 100

print(f"\n性能对比表:")
print(f"{'指标':<20} | {f'模型A ({n_features_without_rfe}特征)':<18} | {'模型B (6特征)':<18} | {'差异':<15}")
print("-" * 85)
print(f"{'训练集 R²':<20} | {metrics_A['train_r2']:>18.6f} | {metrics_B['train_r2']:>18.6f} | {metrics_B['train_r2']-metrics_A['train_r2']:>+15.6f}")
print(f"{'测试集 R²':<20} | {metrics_A['test_r2']:>18.6f} | {metrics_B['test_r2']:>18.6f} | {r2_diff:>+15.6f}")
print(f"{'训练集 RMSE':<20} | {metrics_A['train_rmse']:>18.6f} | {metrics_B['train_rmse']:>18.6f} | {metrics_B['train_rmse']-metrics_A['train_rmse']:>+15.6f}")
print(f"{'测试集 RMSE':<20} | {metrics_A['test_rmse']:>18.6f} | {metrics_B['test_rmse']:>18.6f} | {rmse_diff:>+15.6f}")
print(f"{'训练集 MAE':<20} | {metrics_A['train_mae']:>18.6f} | {metrics_B['train_mae']:>18.6f} | {metrics_B['train_mae']-metrics_A['train_mae']:>+15.6f}")
print(f"{'测试集 MAE':<20} | {metrics_A['test_mae']:>18.6f} | {metrics_B['test_mae']:>18.6f} | {mae_diff:>+15.6f}")
print(f"{'训练时间 (秒)':<20} | {metrics_A['training_time']:>18.2f} | {metrics_B['training_time']:>18.2f} | {metrics_B['training_time']-metrics_A['training_time']:>+15.2f}")
print(f"{'训练轮数':<20} | {metrics_A['final_epoch']:>18d} | {metrics_B['final_epoch']:>18d} | {metrics_B['final_epoch']-metrics_A['final_epoch']:>+15d}")

print(f"\n关键发现:")
print(f"  ✓ 特征数减少: {feature_reduction:.1f}% ({n_features_without_rfe} → 6)")
print(f"  ✓ 测试集R²差异: {r2_diff:+.6f} ({abs(r2_diff/metrics_A['test_r2']*100):.2f}%)")
print(f"  ✓ 测试集RMSE差异: {rmse_diff:+.6f}")

if abs(r2_diff) < 0.02:
    print(f"\n  🎉 结论: RFE非常成功！")
    print(f"     用{6/n_features_without_rfe*100:.1f}%的特征 (6/{n_features_without_rfe}) 获得了相当的性能")
    print(f"     测试集R²仅相差{abs(r2_diff):.4f} (<2%)")
    validation_status = "EXCELLENT"
elif abs(r2_diff) < 0.05:
    print(f"\n  ✓ 结论: RFE有效")
    print(f"     用{6/n_features_without_rfe*100:.1f}%的特征获得了接近的性能")
    print(f"     测试集R²相差{abs(r2_diff):.4f} (<5%)")
    validation_status = "GOOD"
else:
    print(f"\n  ⚠️ 结论: RFE导致一定性能损失")
    print(f"     测试集R²相差{abs(r2_diff):.4f} (>5%)")
    validation_status = "ACCEPTABLE"

# ====================== 10. 数据采样函数 ======================
def sample_history(history, sampling_interval):
    """
    对训练历史数据进行采样
    
    参数:
        history: 原始历史数据字典
        sampling_interval: 采样间隔
    
    返回:
        sampled_history: 采样后的历史数据字典
    """
    sampled_history = {}
    for key in history.keys():
        sampled_history[key] = history[key][::sampling_interval]
    return sampled_history

# ====================== 11. 保存结果 ======================
print("\n" + "="*80)
print("步骤5: 保存结果（✨ 保存全部epoch + 采样数据）")
print("="*80)

# 对历史数据进行采样
print("\n采样训练历史数据...")
history_A_sample50 = sample_history(history_A, 50)
history_B_sample50 = sample_history(history_B, 50)
history_A_sample55 = sample_history(history_A, 55)
history_B_sample55 = sample_history(history_B, 55)

print(f"  模型A - 原始数据点: {len(history_A['epoch'])}")
print(f"  模型A - 每50采样: {len(history_A_sample50['epoch'])} 个数据点")
print(f"  模型A - 每55采样: {len(history_A_sample55['epoch'])} 个数据点")
print(f"  模型B - 原始数据点: {len(history_B['epoch'])}")
print(f"  模型B - 每50采样: {len(history_B_sample50['epoch'])} 个数据点")
print(f"  模型B - 每55采样: {len(history_B_sample55['epoch'])} 个数据点")

# 保存Excel（10个sheet：原4个 + 6个训练历史sheet）
comparison_data = {
    'Metric': [
        'Train R²', 'Test R²', 'Train RMSE', 'Test RMSE', 
        'Train MAE', 'Test MAE', 'Training Time (s)', 'Training Epochs',
        'Number of Features', 'Feature Reduction (%)'
    ],
    'Model A (Without RFE)': [
        f"{metrics_A['train_r2']:.6f}",
        f"{metrics_A['test_r2']:.6f}",
        f"{metrics_A['train_rmse']:.6f}",
        f"{metrics_A['test_rmse']:.6f}",
        f"{metrics_A['train_mae']:.6f}",
        f"{metrics_A['test_mae']:.6f}",
        f"{metrics_A['training_time']:.2f}",
        f"{metrics_A['final_epoch']}",
        f"{n_features_without_rfe}",
        "-"
    ],
    'Model B (With RFE)': [
        f"{metrics_B['train_r2']:.6f}",
        f"{metrics_B['test_r2']:.6f}",
        f"{metrics_B['train_rmse']:.6f}",
        f"{metrics_B['test_rmse']:.6f}",
        f"{metrics_B['train_mae']:.6f}",
        f"{metrics_B['test_mae']:.6f}",
        f"{metrics_B['training_time']:.2f}",
        f"{metrics_B['final_epoch']}",
        "6",
        f"{feature_reduction:.1f}"
    ],
    'Difference (B - A)': [
        f"{metrics_B['train_r2']-metrics_A['train_r2']:+.6f}",
        f"{r2_diff:+.6f}",
        f"{metrics_B['train_rmse']-metrics_A['train_rmse']:+.6f}",
        f"{rmse_diff:+.6f}",
        f"{metrics_B['train_mae']-metrics_A['train_mae']:+.6f}",
        f"{mae_diff:+.6f}",
        f"{metrics_B['training_time']-metrics_A['training_time']:+.2f}",
        f"{metrics_B['final_epoch']-metrics_A['final_epoch']:+d}",
        f"{6-n_features_without_rfe:+d}",
        "-"
    ]
}

excel_path = os.path.join(output_dir, "HTC_performance_comparison_results.xlsx")
with pd.ExcelWriter(excel_path) as writer:
    # Sheet 1: 对比汇总
    pd.DataFrame(comparison_data).to_excel(writer, sheet_name='Comparison_Summary', index=False)
    
    # Sheet 2: 模型A预测值
    pd.DataFrame({
        'Sample_Index': list(range(len(metrics_A['test_true']))),
        'True_Value': metrics_A['test_true'],
        'Predicted_Value': metrics_A['test_pred']
    }).to_excel(writer, sheet_name='Model_A_Predictions', index=False)
    
    # Sheet 3: 模型B预测值
    pd.DataFrame({
        'Sample_Index': list(range(len(metrics_B['test_true']))),
        'True_Value': metrics_B['test_true'],
        'Predicted_Value': metrics_B['test_pred']
    }).to_excel(writer, sheet_name='Model_B_Predictions', index=False)
    
    # Sheet 4: 特征列表
    feature_comparison = pd.DataFrame({
        'Index': list(range(max(len(filtered_features_without_rfe), len(final_features_6)))),
        f'Model A Features ({n_features_without_rfe})': filtered_features_without_rfe + [''] * (max(len(filtered_features_without_rfe), len(final_features_6)) - len(filtered_features_without_rfe)),
        'Model B Features (6)': final_features_6 + [''] * (max(len(filtered_features_without_rfe), len(final_features_6)) - len(final_features_6))
    })
    feature_comparison.to_excel(writer, sheet_name='Feature_Lists', index=False)
    
    # ✨ Sheet 5: 模型A训练历史 - 全部epoch
    history_A_df = pd.DataFrame(history_A)
    history_A_df.to_excel(writer, sheet_name='Model_A_All_Epochs', index=False)
    
    # ✨ Sheet 6: 模型B训练历史 - 全部epoch
    history_B_df = pd.DataFrame(history_B)
    history_B_df.to_excel(writer, sheet_name='Model_B_All_Epochs', index=False)
    
    # ✨ Sheet 7: 模型A训练历史 - 每50个epoch采样
    history_A_sample50_df = pd.DataFrame(history_A_sample50)
    history_A_sample50_df.to_excel(writer, sheet_name='Model_A_Sample_50', index=False)
    
    # ✨ Sheet 8: 模型B训练历史 - 每50个epoch采样
    history_B_sample50_df = pd.DataFrame(history_B_sample50)
    history_B_sample50_df.to_excel(writer, sheet_name='Model_B_Sample_50', index=False)
    
    # ✨ Sheet 9: 模型A训练历史 - 每55个epoch采样
    history_A_sample55_df = pd.DataFrame(history_A_sample55)
    history_A_sample55_df.to_excel(writer, sheet_name='Model_A_Sample_55', index=False)
    
    # ✨ Sheet 10: 模型B训练历史 - 每55个epoch采样
    history_B_sample55_df = pd.DataFrame(history_B_sample55)
    history_B_sample55_df.to_excel(writer, sheet_name='Model_B_Sample_55', index=False)

print(f"\n✅ Excel结果已保存 (10个sheet): {excel_path}")
print(f"   Sheet 1: Comparison_Summary - 性能对比汇总")
print(f"   Sheet 2: Model_A_Predictions - 模型A预测值")
print(f"   Sheet 3: Model_B_Predictions - 模型B预测值")
print(f"   Sheet 4: Feature_Lists - 特征列表")
print(f"   ✨ Sheet 5: Model_A_All_Epochs - 模型A全部epoch ({len(history_A['epoch'])}点)")
print(f"   ✨ Sheet 6: Model_B_All_Epochs - 模型B全部epoch ({len(history_B['epoch'])}点)")
print(f"   ✨ Sheet 7: Model_A_Sample_50 - 模型A每50采样 ({len(history_A_sample50['epoch'])}点)")
print(f"   ✨ Sheet 8: Model_B_Sample_50 - 模型B每50采样 ({len(history_B_sample50['epoch'])}点)")
print(f"   ✨ Sheet 9: Model_A_Sample_55 - 模型A每55采样 ({len(history_A_sample55['epoch'])}点)")
print(f"   ✨ Sheet 10: Model_B_Sample_55 - 模型B每55采样 ({len(history_B_sample55['epoch'])}点)")

# ====================== 12. 可视化（使用采样数据绘图以提高性能）======================
print("\n" + "="*80)
print("步骤6: 生成可视化图表（✨ 使用每50采样数据绘图）")
print("="*80)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 图1: 性能对比柱状图
fig1, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics_names = ['R²', 'RMSE', 'MAE']
model_A_values = [metrics_A['test_r2'], metrics_A['test_rmse'], metrics_A['test_mae']]
model_B_values = [metrics_B['test_r2'], metrics_B['test_rmse'], metrics_B['test_mae']]

x = np.arange(len(metrics_names))
width = 0.35

for idx, (ax, metric) in enumerate(zip(axes, metrics_names)):
    bars1 = ax.bar(x[idx] - width/2, model_A_values[idx], width, 
                   label=f'Model A ({n_features_without_rfe} features)', color='steelblue', alpha=0.8)
    bars2 = ax.bar(x[idx] + width/2, model_B_values[idx], width,
                   label='Model B (6 features)', color='coral', alpha=0.8)
    
    ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'Test {metric} Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks([0])
    ax.set_xticklabels([metric])
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.4f}', ha='center', va='bottom', fontsize=10)
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.4f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('HTC Model Performance Comparison: With vs Without RFE', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig1_path = os.path.join(output_dir, "1_HTC_performance_comparison_barplot.png")
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
print(f"✅ 图1已保存: {fig1_path}")
plt.close()

# 图2: 散点图
fig2, axes = plt.subplots(1, 2, figsize=(16, 7))

ax1 = axes[0]
all_values_A = np.concatenate([metrics_A['test_true'], metrics_A['test_pred']])
min_val_A = np.min(all_values_A)
max_val_A = np.max(all_values_A)

ax1.plot([min_val_A, max_val_A], [min_val_A, max_val_A], 'k--', alpha=0.75)
ax1.scatter(metrics_A['test_true'], metrics_A['test_pred'], 
           color='steelblue', alpha=0.6, s=50, edgecolor='black', linewidth=0.5)
ax1.set_xlabel('True Value (MPa)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Predicted Value (MPa)', fontsize=12, fontweight='bold')
ax1.set_title(f"Model A ({n_features_without_rfe} features)\nTest R²={metrics_A['test_r2']:.4f}, RMSE={metrics_A['test_rmse']:.4f}", 
             fontsize=13, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.set_aspect('equal')

ax2 = axes[1]
all_values_B = np.concatenate([metrics_B['test_true'], metrics_B['test_pred']])
min_val_B = np.min(all_values_B)
max_val_B = np.max(all_values_B)

ax2.plot([min_val_B, max_val_B], [min_val_B, max_val_B], 'k--', alpha=0.75)
ax2.scatter(metrics_B['test_true'], metrics_B['test_pred'], 
           color='coral', alpha=0.6, s=50, edgecolor='black', linewidth=0.5)
ax2.set_xlabel('True Value (MPa)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Predicted Value (MPa)', fontsize=12, fontweight='bold')
ax2.set_title(f"Model B (6 features)\nTest R²={metrics_B['test_r2']:.4f}, RMSE={metrics_B['test_rmse']:.4f}", 
             fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)
ax2.set_aspect('equal')

plt.suptitle('HTC Model Prediction Scatter Plots: With vs Without RFE', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
fig2_path = os.path.join(output_dir, "2_HTC_prediction_scatterplots.png")
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
print(f"✅ 图2已保存: {fig2_path}")
plt.close()

# 图3: 训练曲线图（R², RMSE, MAE）- 使用采样数据
fig3, axes = plt.subplots(3, 1, figsize=(14, 12))

# 子图1: R²曲线
ax1 = axes[0]
ax1.plot(history_A_sample50['epoch'], history_A_sample50['train_r2'], 
        label=f'Model A ({n_features_without_rfe} feat) - Train', 
        color='steelblue', linewidth=2, alpha=0.8)
ax1.plot(history_A_sample50['epoch'], history_A_sample50['test_r2'], 
        label=f'Model A ({n_features_without_rfe} feat) - Test', 
        color='steelblue', linewidth=2, alpha=0.8, linestyle='--')
ax1.plot(history_B_sample50['epoch'], history_B_sample50['train_r2'], 
        label='Model B (6 feat) - Train', 
        color='coral', linewidth=2, alpha=0.8)
ax1.plot(history_B_sample50['epoch'], history_B_sample50['test_r2'], 
        label='Model B (6 feat) - Test', 
        color='coral', linewidth=2, alpha=0.8, linestyle='--')
ax1.axhline(y=0.92, color='green', linestyle=':', linewidth=1.5, alpha=0.5, label='Target (0.92)')
ax1.set_ylabel('R²', fontsize=13, fontweight='bold')
ax1.set_title('Training Progress: R² Score (Sampled every 50 epochs)', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=10)
ax1.grid(alpha=0.3)

# 子图2: RMSE曲线
ax2 = axes[1]
ax2.plot(history_A_sample50['epoch'], history_A_sample50['train_rmse'], 
        label=f'Model A ({n_features_without_rfe} feat) - Train', 
        color='steelblue', linewidth=2, alpha=0.8)
ax2.plot(history_A_sample50['epoch'], history_A_sample50['test_rmse'], 
        label=f'Model A ({n_features_without_rfe} feat) - Test', 
        color='steelblue', linewidth=2, alpha=0.8, linestyle='--')
ax2.plot(history_B_sample50['epoch'], history_B_sample50['train_rmse'], 
        label='Model B (6 feat) - Train', 
        color='coral', linewidth=2, alpha=0.8)
ax2.plot(history_B_sample50['epoch'], history_B_sample50['test_rmse'], 
        label='Model B (6 feat) - Test', 
        color='coral', linewidth=2, alpha=0.8, linestyle='--')
ax2.axhline(y=6.0, color='orange', linestyle=':', linewidth=1.5, alpha=0.5, label='Target (6.0)')
ax2.set_ylabel('RMSE (MPa)', fontsize=13, fontweight='bold')
ax2.set_title('Training Progress: RMSE (Sampled every 50 epochs)', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(alpha=0.3)

# 子图3: MAE曲线
ax3 = axes[2]
ax3.plot(history_A_sample50['epoch'], history_A_sample50['train_mae'], 
        label=f'Model A ({n_features_without_rfe} feat) - Train', 
        color='steelblue', linewidth=2, alpha=0.8)
ax3.plot(history_A_sample50['epoch'], history_A_sample50['test_mae'], 
        label=f'Model A ({n_features_without_rfe} feat) - Test', 
        color='steelblue', linewidth=2, alpha=0.8, linestyle='--')
ax3.plot(history_B_sample50['epoch'], history_B_sample50['train_mae'], 
        label='Model B (6 feat) - Train', 
        color='coral', linewidth=2, alpha=0.8)
ax3.plot(history_B_sample50['epoch'], history_B_sample50['test_mae'], 
        label='Model B (6 feat) - Test', 
        color='coral', linewidth=2, alpha=0.8, linestyle='--')
ax3.set_xlabel('Epoch', fontsize=13, fontweight='bold')
ax3.set_ylabel('MAE (MPa)', fontsize=13, fontweight='bold')
ax3.set_title('Training Progress: MAE (Sampled every 50 epochs)', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right', fontsize=10)
ax3.grid(alpha=0.3)

plt.suptitle('HTC Model Training Curves: With vs Without RFE', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
fig3_path = os.path.join(output_dir, "3_HTC_training_curves_combined.png")
plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
print(f"✨ 图3已保存（R², RMSE, MAE曲线）: {fig3_path}")
plt.close()

# 图4: Loss曲线图 - 使用采样数据
fig4, ax = plt.subplots(figsize=(14, 6))

ax.plot(history_A_sample50['epoch'], history_A_sample50['train_loss'], 
        label=f'Model A ({n_features_without_rfe} feat) - Train Loss', 
        color='steelblue', linewidth=2, alpha=0.8)
ax.plot(history_A_sample50['epoch'], history_A_sample50['test_loss'], 
        label=f'Model A ({n_features_without_rfe} feat) - Test Loss', 
        color='steelblue', linewidth=2, alpha=0.8, linestyle='--')
ax.plot(history_B_sample50['epoch'], history_B_sample50['train_loss'], 
        label='Model B (6 feat) - Train Loss', 
        color='coral', linewidth=2, alpha=0.8)
ax.plot(history_B_sample50['epoch'], history_B_sample50['test_loss'], 
        label='Model B (6 feat) - Test Loss', 
        color='coral', linewidth=2, alpha=0.8, linestyle='--')

ax.set_xlabel('Epoch', fontsize=13, fontweight='bold')
ax.set_ylabel('Loss (Combined MSE+Huber)', fontsize=13, fontweight='bold')
ax.set_title('HTC Model Training Progress: Loss Curves (Sampled every 50 epochs)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
fig4_path = os.path.join(output_dir, "4_HTC_loss_curves.png")
plt.savefig(fig4_path, dpi=300, bbox_inches='tight')
print(f"✨ 图4已保存（Loss曲线）: {fig4_path}")
plt.close()

# ====================== 13. 保存报告 ======================
report_path = os.path.join(output_dir, "HTC_performance_comparison_report.txt")
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("HTC模型性能对比验证报告\n")
    f.write("Performance Comparison: With vs Without RFE (HTC Model)\n")
    f.write("✨ 公平epoch对比 + 完整训练历史数据（含Loss）\n")
    f.write("✨ 模型B最大训练上限：1000000 epoch\n")
    f.write("✨ Excel保存：全部epoch + 每50采样 + 每55采样\n")
    f.write("="*80 + "\n\n")
    
    f.write("1. 验证目的\n")
    f.write("-" * 80 + "\n")
    f.write("通过训练两个HTC模型验证RFE特征选择的有效性\n\n")
    
    f.write("2. 模型配置\n")
    f.write("-" * 80 + "\n")
    f.write("训练策略:\n")
    f.write("  1. 先训练模型B (With RFE - 6特征)，最大上限1000000 epoch\n")
    f.write(f"  2. 模型B在epoch {metrics_B['final_epoch']} 达到目标\n")
    f.write(f"  3. 然后训练模型A到相同的epoch {metrics_A['final_epoch']}\n")
    f.write("  4. 确保两个模型的epoch数完全相同，实现公平对比\n\n")
    f.write("模型结构（HTC架构 - 两个模型相同）:\n")
    f.write("  - 输入层 → 隐藏层1: 32神经元 (Linear + BatchNorm + ReLU + Dropout 0.05)\n")
    f.write("  - 隐藏层1 → 隐藏层2: 16神经元 (Linear + BatchNorm + ReLU + Dropout 0.02)\n")
    f.write("  - 隐藏层2 → 输出层: 1神经元\n")
    f.write("  - 损失函数: Combined Loss (0.6*MSE + 0.4*Huber)\n")
    f.write("  - 优化器: Adam (lr=0.0015, weight_decay=1e-6)\n")
    f.write("  - 学习率调度: WarmupCosine (warmup=200步)\n\n")
    
    f.write(f"3. 模型A (Without RFE - {n_features_without_rfe} Features)\n")
    f.write("-" * 80 + "\n")
    f.write(f"特征数: {n_features_without_rfe}\n")
    f.write(f"训练集 R²: {metrics_A['train_r2']:.6f}\n")
    f.write(f"测试集 R²: {metrics_A['test_r2']:.6f}\n")
    f.write(f"训练集 RMSE: {metrics_A['train_rmse']:.6f}\n")
    f.write(f"测试集 RMSE: {metrics_A['test_rmse']:.6f}\n")
    f.write(f"训练轮数: {metrics_A['final_epoch']}\n")
    f.write(f"训练历史数据点:\n")
    f.write(f"  - 全部epoch: {len(history_A['epoch'])}\n")
    f.write(f"  - 每50采样: {len(history_A_sample50['epoch'])}\n")
    f.write(f"  - 每55采样: {len(history_A_sample55['epoch'])}\n\n")
    
    f.write("4. 模型B (With RFE - 6 Features)\n")
    f.write("-" * 80 + "\n")
    f.write(f"特征数: 6\n")
    for i, feat in enumerate(final_features_6, 1):
        f.write(f"  {i}. {feat}\n")
    f.write(f"\n训练集 R²: {metrics_B['train_r2']:.6f}\n")
    f.write(f"测试集 R²: {metrics_B['test_r2']:.6f}\n")
    f.write(f"训练集 RMSE: {metrics_B['train_rmse']:.6f}\n")
    f.write(f"测试集 RMSE: {metrics_B['test_rmse']:.6f}\n")
    f.write(f"训练轮数: {metrics_B['final_epoch']}\n")
    f.write(f"训练历史数据点:\n")
    f.write(f"  - 全部epoch: {len(history_B['epoch'])}\n")
    f.write(f"  - 每50采样: {len(history_B_sample50['epoch'])}\n")
    f.write(f"  - 每55采样: {len(history_B_sample55['epoch'])}\n\n")
    
    f.write("5. 性能差异分析\n")
    f.write("-" * 80 + "\n")
    f.write(f"特征数减少: {feature_reduction:.1f}% ({n_features_without_rfe} → 6)\n")
    f.write(f"测试集R²差异: {r2_diff:+.6f}\n")
    f.write(f"测试集RMSE差异: {rmse_diff:+.6f}\n")
    f.write(f"测试集MAE差异: {mae_diff:+.6f}\n\n")
    
    f.write("6. 输出文件清单\n")
    f.write("-" * 80 + "\n")
    f.write("Excel文件 (10个sheet):\n")
    f.write("  1. Comparison_Summary - 性能对比汇总\n")
    f.write("  2. Model_A_Predictions - 模型A预测值\n")
    f.write("  3. Model_B_Predictions - 模型B预测值\n")
    f.write("  4. Feature_Lists - 特征列表\n")
    f.write(f"  ✨ 5. Model_A_All_Epochs - 模型A全部epoch ({len(history_A['epoch'])}点)\n")
    f.write(f"  ✨ 6. Model_B_All_Epochs - 模型B全部epoch ({len(history_B['epoch'])}点)\n")
    f.write(f"  ✨ 7. Model_A_Sample_50 - 模型A每50采样 ({len(history_A_sample50['epoch'])}点)\n")
    f.write(f"  ✨ 8. Model_B_Sample_50 - 模型B每50采样 ({len(history_B_sample50['epoch'])}点)\n")
    f.write(f"  ✨ 9. Model_A_Sample_55 - 模型A每55采样 ({len(history_A_sample55['epoch'])}点)\n")
    f.write(f"  ✨ 10. Model_B_Sample_55 - 模型B每55采样 ({len(history_B_sample55['epoch'])}点)\n\n")
    f.write("可视化图片:\n")
    f.write("  1. HTC_performance_comparison_barplot.png - 性能对比柱状图\n")
    f.write("  2. HTC_prediction_scatterplots.png - 预测散点图\n")
    f.write("  3. HTC_training_curves_combined.png - 训练曲线图（R², RMSE, MAE，每50采样）\n")
    f.write("  ✨ 4. HTC_loss_curves.png - Loss曲线图（训练/测试Loss，每50采样）\n\n")
    
    f.write("7. 结论\n")
    f.write("-" * 80 + "\n")
    f.write(f"验证状态: {validation_status}\n\n")
    
    if validation_status == "EXCELLENT":
        f.write("✓✓✓ RFE特征选择非常成功！\n\n")
        f.write(f"使用{6/n_features_without_rfe*100:.1f}%的特征获得了相当性能\n")
    elif validation_status == "GOOD":
        f.write("✓✓ RFE特征选择有效\n\n")
    
    f.write("="*80 + "\n")

print(f"✅ 验证报告已保存: {report_path}")

# 保存模型
model_A_path = os.path.join(output_dir, f"HTC_model_A_{n_features_without_rfe}features.pth")
torch.save(model_A.state_dict(), model_A_path)
print(f"✅ 模型A已保存: {model_A_path}")

model_B_path = os.path.join(output_dir, "HTC_model_B_6features.pth")
torch.save(model_B.state_dict(), model_B_path)
print(f"✅ 模型B已保存: {model_B_path}")

# ====================== 14. 总结 ======================
print("\n" + "="*80)
print("✨ HTC模型分析完成！(公平epoch对比 + Loss曲线完整版 + 采样数据)")
print("✨ 模型B最大训练上限：1000000 epoch")
print("✨ Excel保存：全部epoch + 每50采样 + 每55采样")
print("="*80)
print(f"\n所有结果已保存到: {output_dir}")
print("\n📊 输出文件清单:")
print("   1个Excel文件 (10个sheet)")
print("     - 4个基础sheet（对比汇总、预测值、特征列表）")
print("     - 6个训练历史sheet（全部epoch + 每50采样 + 每55采样）")
print("   4张可视化图片（使用每50采样数据绘制）")
print("   1份文本报告")
print("   2个模型文件")

print("\n✨ 核心特性:")
print(f"   ✅ 模型B (With RFE) 先训练: 在epoch {metrics_B['final_epoch']} 达到目标 (最大上限1000000)")
print(f"   ✅ 模型A (Without RFE) 训练到相同epoch: {metrics_A['final_epoch']}")
print(f"   ✅ 公平对比: 两个模型训练轮数相同")
print(f"   ✅ 完整数据: 每个epoch的 R², RMSE, MAE, Loss")
print(f"   ✨ 多种采样: 全部epoch ({len(history_A['epoch'])}) + 每50采样 ({len(history_A_sample50['epoch'])}) + 每55采样 ({len(history_A_sample55['epoch'])})")
print("   ✅ Loss曲线可视化图（训练/测试Loss）")
print("   ✅ 所有数据可直接用于Origin绘制高质量曲线")

print("\n📈 关键发现:")
print(f"   验证状态: {validation_status}")
print(f"   模型A ({n_features_without_rfe}特征): Test R² = {metrics_A['test_r2']:.6f}")
print(f"   模型B (6特征):  Test R² = {metrics_B['test_r2']:.6f}")
print(f"   性能差异: {r2_diff:+.6f}")
print(f"   特征减少: {feature_reduction:.1f}%")

if validation_status == "EXCELLENT":
    print(f"\n🎉 恭喜！RFE验证完全通过！")
elif validation_status == "GOOD":
    print(f"\n✓ RFE验证通过！")

print("\n💡 Origin绘图说明:")
print("   - 打开Excel文件")
print("   - 根据数据量选择合适的sheet:")
print("     * 高精度分析: 使用Sheet 5/6 (全部epoch)")
print("     * 常规绘图: 使用Sheet 7/8 (每50采样，推荐)")
print("     * 备选方案: 使用Sheet 9/10 (每55采样)")
print("   - 使用epoch列作为X轴")
print("   - 绘制以下曲线：")
print("     * train_r2, test_r2（R²曲线）")
print("     * train_rmse, test_rmse（RMSE曲线）")
print("     * train_mae, test_mae（MAE曲线）")
print("     ✨ * train_loss, test_loss（Loss曲线）")

print("="*80)

HTC模型性能对比验证：With vs Without RFE
✨ 公平epoch对比 + 完整训练历史数据（含Loss）
✨ 模型B最大训练上限：1000000 epoch
✨ Excel保存：全部epoch + 每50采样 + 每55采样

原始数据信息:
  样本数: 34
  原始特征数: 96
  目标变量范围: [190.0000, 588.6000]

创建输出目录: D:\博一下\manuscript3-机器学习\nc\nc修改9.27\10.15\10.21\带不带RFE\HTC_performance_comparison-1.25.2

步骤1: 特征选择流程（获得Without RFE特征和6特征）

[1/3] PCC去除冗余特征...
  去冗余前: 96 个特征
  去冗余后: 35 个特征

[2/3] PCC + MIC 过滤...
  PCC+MIC过滤后: 7 个特征
  这就是 'Without RFE' 的特征集

[3/3] RFE选择6个特征...
  RFE选择后: 6 个特征
  这就是 'With RFE' 的6个特征
  特征名称: ['am', 'Mean_Speed of Sound', 'Mean_热膨胀(1/k)', 'Mean_比热容J/g/k', 'Mean_Pyykko(Triple Bond) (pm)', 'Var_E13 Electron affinity']

✅ 特征选择完成！
   模型A (Without RFE): 7 个特征
   模型B (With RFE): 6 个特征

步骤2: 准备两组数据并标准化

模型A数据 (Without RFE):
  特征数: 7
  样本数: 34

模型B数据 (With RFE):
  特征数: 6
  样本数: 34

步骤3: 训练两个模型（✨ 先训练With RFE，然后用相同epoch训练Without RFE）

第一步：训练 Model B (With RFE - 6 features)

--------------------------------------------------------------------------------
训练 Model B (With RFE - 6 features)
最大训练